# 10 — Bridge Table (M:N) — Spark SQL

Bridge Table Employee ↔ Territory via `INSERT INTO ... SELECT DISTINCT`.

**Técnica:** `CREATE OR REPLACE TEMP VIEW` com JOIN + DISTINCT → `TRUNCATE` + `INSERT INTO`.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 10 Bridge Table")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:57:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_bridge AS
    SELECT DISTINCT de.EmployeeSK, dt.TerritorySK
    FROM bronze.employee_territories et
    JOIN gold.DimEmployee  de ON et.EmployeeID  = de.EmployeeID
    JOIN gold.DimTerritory dt ON et.TerritoryID = dt.TerritoryID
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.BridgeEmployeeTerritory
    (EmployeeSK INT, TerritorySK INT) USING DELTA
""")
spark.sql("DELETE FROM gold.BridgeEmployeeTerritory")
spark.sql("INSERT INTO gold.BridgeEmployeeTerritory SELECT EmployeeSK, TerritorySK FROM src_bridge")

count = spark.sql("SELECT COUNT(*) AS n FROM gold.BridgeEmployeeTerritory").collect()[0]["n"]
print(f"Bridge carregada: {count} linhas")
assert count == 49, f"Esperado 49, got {count}"
print("Bridge completude OK")

Bridge carregada: 49 linhas
Bridge completude OK


In [4]:
# DEMO 1: Receita por território via bridge
spark.sql("""
    SELECT dt.RegionName, dt.TerritoryDescription,
           COUNT(DISTINCT fs.OrderID) AS OrderCount,
           ROUND(SUM(fs.NetRevenue), 2) AS TotalRevenue
    FROM gold.FactSales fs
    JOIN gold.DimEmployee de ON de.EmployeeSK = fs.EmployeeSK
    JOIN gold.BridgeEmployeeTerritory b ON b.EmployeeSK = de.EmployeeSK
    JOIN gold.DimTerritory dt ON dt.TerritorySK = b.TerritorySK
    GROUP BY dt.RegionName, dt.TerritoryDescription
    ORDER BY TotalRevenue DESC LIMIT 10
""").show(truncate=False)

+--------------------------------------------------+--------------------------------------------------+----------+------------+
|RegionName                                        |TerritoryDescription                              |OrderCount|TotalRevenue|
+--------------------------------------------------+--------------------------------------------------+----------+------------+
|Eastern                                           |Greensboro                                        |156       |232890.85   |
|Eastern                                           |Rockville                                         |156       |232890.85   |
|Eastern                                           |Cary                                              |156       |232890.85   |
|Southern                                          |Tampa                                             |127       |202812.84   |
|Southern                                          |Atlanta                                           |1

In [5]:
# DEMO 2: Empregados com mais territórios
spark.sql("""
    SELECT de.FullName, COUNT(b.TerritorySK) AS QuantidadeTerretorios
    FROM gold.BridgeEmployeeTerritory b
    JOIN gold.DimEmployee de ON de.EmployeeSK = b.EmployeeSK
    GROUP BY de.FullName ORDER BY QuantidadeTerretorios DESC
""").show()

+----------------+---------------------+
|        FullName|QuantidadeTerretorios|
+----------------+---------------------+
|     Robert King|                   10|
|   Andrew Fuller|                    7|
| Steven Buchanan|                    7|
|  Anne Dodsworth|                    7|
|  Michael Suyama|                    5|
| Janet Leverling|                    4|
|  Laura Callahan|                    4|
|Margaret Peacock|                    3|
|   Nancy Davolio|                    2|
+----------------+---------------------+

